In [1]:
# Imports
import sys
import os

# Add the upstream directory to sys.path
upstream_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if upstream_dir not in sys.path:
    sys.path.insert(0, upstream_dir)

# Now you can import the module
from opentrons_api import ot2_api
from microtissue_manipulator import core, utils, camera
import numpy as np 
import cv2
import time
import json
import keyboard
# from pynput import keyboard
import paths
import matplotlib.pyplot as plt
import requests
import datetime
import threading
import queue
import string
import pandas as pd
import multiprocessing as mp
from dataclasses import dataclass, fields, asdict, MISSING
from typing import get_type_hints, get_origin, get_args, Tuple, List, Dict, Any, Union
from ultralytics import YOLO
from enum import Enum

# Connect camera

In [3]:
cam_manager = camera.CameraManagerWindows()
print("Connected cameras:")
print(cam_manager.list_devices())

microscope_cam = camera.open_capture('microscope_cam', cam_manager=cam_manager)

Connected cameras:
['HD USB CAMERA', 'Digital Microscope']


In [4]:
microscope_cam.read()

(True,
 array([[[ 7,  4, 17],
         [ 6,  3, 16],
         [ 5,  2, 15],
         ...,
         [ 0, 22, 58],
         [ 4, 25, 63],
         [ 9, 28, 68]],
 
        [[ 9,  5, 19],
         [ 8,  4, 18],
         [ 8,  4, 18],
         ...,
         [ 0, 19, 57],
         [ 0, 20, 61],
         [ 0, 21, 63]],
 
        [[10,  6, 21],
         [10,  6, 21],
         [10,  6, 21],
         ...,
         [ 0, 27, 68],
         [ 1, 26, 70],
         [ 1, 25, 70]],
 
        ...,
 
        [[ 1,  0,  2],
         [ 1,  0,  2],
         [ 1,  0,  2],
         ...,
         [ 3, 10, 29],
         [ 3, 10, 30],
         [ 3, 10, 30]],
 
        [[ 1,  0,  2],
         [ 1,  0,  2],
         [ 1,  0,  2],
         ...,
         [ 2,  9, 29],
         [ 1,  7, 28],
         [ 1,  7, 29]],
 
        [[ 1,  0,  2],
         [ 1,  0,  2],
         [ 1,  0,  2],
         ...,
         [ 1,  8, 28],
         [ 0,  6, 28],
         [ 0,  6, 28]]], dtype=uint8))

# Init robot

In [5]:
# Connect the robot to the computer and this notebook
openapi = ot2_api.OpentronsAPI()

In [6]:
# Set the offset if the platform is used
openapi.add_slot_offsets([5,8,9], (0,0,64.2))

In [7]:
# Use the light control to see if the robot is connected as a sanity check
openapi.toggle_lights()

<Response [200]>

In [8]:
# Always do once after robot was just turned on
openapi.home_robot()

Request status:
<Response [200]>
{
  "message": "Homing robot."
}


<Response [200]>

In [9]:
# Use to restore labware and general run information after the notebook crashes
r = openapi.get_run_info()

Total number of runs: 20
Current run ID: None
Current run status: None


In [10]:
# Do after first launch
openapi.create_run()

Request status:
<Response [201]>
{
  "data": {
    "id": "c9288631-f38d-4a01-a804-bc450622865e",
    "ok": true,
    "createdAt": "2025-06-03T17:00:08.926765Z",
    "status": "idle",
    "current": true,
    "actions": [],
    "errors": [],
    "hasEverEnteredErrorRecovery": false,
    "pipettes": [],
    "modules": [],
    "labware": [],
    "liquids": [],
    "liquidClasses": [],
    "labwareOffsets": [],
    "runTimeParameters": [],
    "outputFileIds": []
  }
}


<Response [201]>

In [11]:
# Let the robot know that it has the P300 pipette
openapi.load_pipette()

Request status:
<Response [201]>
{
  "data": {
    "id": "f181d85d-74c9-40c4-a482-000b178be54f",
    "createdAt": "2025-06-03T17:00:14.561157Z",
    "commandType": "loadPipette",
    "key": "f181d85d-74c9-40c4-a482-000b178be54f",
    "status": "succeeded",
    "params": {
      "pipetteName": "p300_single_gen2",
      "mount": "left"
    },
    "result": {
      "pipetteId": "a4e0d0d5-7920-45ae-8098-bb6e74fd7302"
    },
    "startedAt": "2025-06-03T17:00:14.564371Z",
    "completedAt": "2025-06-03T17:00:16.476767Z",
    "intent": "setup",
    "notes": []
  }
}


<Response [201]>

In [12]:
openapi.move_labware(openapi.labware_dct['5'], 'offDeck')

Exception: {"errors":[{"id":"InvalidRequest","title":"Invalid Request","detail":"Input should be a valid string","source":{"pointer":"/data/moveLabware/params/labwareId"},"errorCode":"4000"}]}

# Labware declaration

### Opentrons 300 ul

In [13]:
#Define a tip rack. This is the default tip rack for the robot.
TIP_RACK = "opentrons_96_tiprack_300ul"
#Load the tip rack. Slot = 1 by default.
r = openapi.load_labware(TIP_RACK, 11)

Labware ID:
d0d9b6e0-e8aa-43bc-8778-bc0880c3357f



In [14]:
r = openapi.pick_up_tip(openapi.labware_dct['11'], "A2")

### Well plate

In [19]:
WELL_PLATE = "corning_96_wellplate_360ul_flat"
# WELL_PLATE = "corning_24_wellplate_3.4ml_flat"
# WELL_PLATE = "corning_384_wellplate_112ul_flat"
r = openapi.load_labware(WELL_PLATE, 5, namespace='opentrons',verbose=True)

Offset (0, 0, 64.2) added to run for corning_96_wellplate_360ul_flat in slot 5.
Labware URI:
opentrons/corning_96_wellplate_360ul_flat/1

Check offset before using ...


AssertionError: Error loading labware...

# Slice picking

In [26]:
openapi.toggle_lights()

<Response [200]>

In [21]:
openapi.retract_axis('leftZ')
openapi.move_to_well(openapi.labware_dct['5'], 'A1', well_location='top', offset=(0,0,1))

<Response [201]>

In [16]:
openapi.retract_axis('leftZ')
openapi.move_to_coordinates((300, 135, 110), min_z_height=1, verbose=False)

<Response [201]>

In [60]:
def on_mouse_click(event, cX, cY, flags, param):
    global X_init, Y_init, X, Y, target_x, target_y

    if event == cv2.EVENT_RBUTTONDOWN:
        x, y, _ = openapi.get_position(verbose=False)[0].values()
        openapi.retract_axis('leftZ')
        openapi.move_to_coordinates((300, 135, 110), min_z_height=1, verbose=False)

    if event == cv2.EVENT_MOUSEWHEEL:
        # List of well names in 96-well plate (A1-H12)
        if not hasattr(on_mouse_click, "well_index"):
            on_mouse_click.well_index = 0
        if not hasattr(on_mouse_click, "well_names"):
            rows = "ABCDEFGH"
            cols = range(1, 13)
            on_mouse_click.well_names = [f"{row}{col}" for row in rows for col in cols]
        num_wells = len(on_mouse_click.well_names)
        # event flags: positive for scroll up, negative for scroll down
        if flags > 0:
            on_mouse_click.well_index = (on_mouse_click.well_index + 1) % num_wells
        else:
            on_mouse_click.well_index = (on_mouse_click.well_index - 1) % num_wells
        current_well = on_mouse_click.well_names[on_mouse_click.well_index]
        # print(f"Selected well: {current_well}")


# Create an instance of the ManualRobotMovement class
manual_movement = utils.ManualRobotMovement(openapi)
cv2.namedWindow("video", cv2.WINDOW_NORMAL)
cv2.resizeWindow("video", 1348, 1011)
# cv2.resizeWindow("video", 1050, 1348)
cv2.setMouseCallback("video", on_mouse_click)

target_x, target_y = 0, 0

dish_bottom = 70# - 11.5
pickup_offset = 0.0 #0.6
flow_rate = 75
volume = 10

x_asp, y_asp, z_asp = (300, 135, 110)

while True:
    ret, frame = microscope_cam.read()
    frame = cv2.flip(frame, 1)
    cv2.circle(frame, (target_x, target_y), 3, (0, 0, 255), -1)
    x, y, z = openapi.get_position(verbose=False)[0].values()
    cv2.putText(frame, f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Step size: {manual_movement.step} mm", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Flow rate: {flow_rate} uL/s", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Volume: {volume} uL", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    if hasattr(on_mouse_click, "well_names") and hasattr(on_mouse_click, "well_index"):
        current_well = on_mouse_click.well_names[on_mouse_click.well_index]
        cv2.putText(frame, f"Current well: {current_well}", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("video", frame)

    # Draw a point at the center of the screen
    key_pressed = cv2.waitKey(1)

    if key_pressed == ord('q'):
        keyboard.unhook_all()
        break

    elif key_pressed == ord('d'):
        openapi.dispense_in_place(flow_rate = flow_rate, volume = volume)

    elif key_pressed == ord('b'):
        openapi.blow_out_in_place(50)

    elif key_pressed == ord('a'):
        x_asp, y_asp, z_asp = openapi.get_position(verbose=False)[0].values()
        openapi.aspirate_in_place(flow_rate = flow_rate, volume = volume, verbose=True)

    elif key_pressed == ord('r'):
        openapi.retract_axis('leftZ')
        openapi.move_to_coordinates((x_asp, y_asp, z_asp + 10), min_z_height=1, verbose=False)

    elif key_pressed == ord('g'):
        if current_well:
            openapi.retract_axis('leftZ')
            openapi.move_to_well(openapi.labware_dct['5'], well_name=current_well, well_location='top', offset = (0,0,0), min_z_height=70, verbose=False)

    elif key_pressed == ord('s'):
        timestamp = datetime.datetime.now().strftime('%Y_%m_%d_%H-%M-%S')
        filename = f"frame_{timestamp}.png"
        cv2.imwrite(str(paths.BASE_DIR)+'\\outputs\\images\\2025-09-10_slice_test\\'+filename, frame)
        print(f"Frame saved as {filename}")

cv2.destroyAllWindows()

Frame saved as frame_2025_09_17_12-46-58.png
